In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pickle
import os
import sys
from tqdm.contrib.concurrent import process_map  # multiprocessing + tqdm
from functools import partial
import multiprocessing

from concurrent.futures import ProcessPoolExecutor, as_completed
sys.path.append("../../../src/irl")
import reddit_traj_construction



In [ ]:
with open("./config.json") as f:
    config = json.load(f)
    collection_start_date = int(config["collection_start_date"])
    collection_end_date = int(config["collection_end_date"])
    root_output_dir_path = config["root_output_dir_path"]
    deberta_weight_path = config["deberta_weight_path"]
    
    processed_per_user_path =  root_output_dir_path + "/4_per_user_processing_collated" 


load_dir = processed_per_user_path
output_dir = root_output_dir_path + "/user_content"
os.makedirs(output_dir, exist_ok=True)

all_users = sorted(os.listdir(load_dir))
print(len(all_users))


In [ ]:
# rus active timeframe 
start_date = "2015-01-02"
end_date = "2018-04-11"

# Helper function to run in parallel
def get_user_df(user):
    df_slice = reddit_traj_construction.get_user_activity_in_timeframe(
        user,
        load_dir,
        start_date=start_date,
        end_date=end_date
    )
    # Only keep relevant rows and columns
    filtered = df_slice[df_slice.action.isin([1, 2, 3])].copy()

    # Ensure column exists
    if 'agree_prediction' not in filtered.columns:
        filtered['agree_prediction'] = None
    if 'url' not in filtered.columns:
        filtered['url'] = None
    if 'title' not in filtered.columns:
        filtered['title'] = None
    if 'selftext' not in filtered.columns:
        filtered['selftext'] = None
    if 'subreddit' not in filtered.columns:
        filtered['subreddit'] = None

    return filtered[["author", "id", "created_utc", "subreddit", "body", "selftext", "action", "url", "title", 'agree_prediction']].copy()

# Use all available cores
num_workers = multiprocessing.cpu_count()

df_list = []
with ProcessPoolExecutor(max_workers=num_workers) as executor:
    futures = {executor.submit(get_user_df, user): user for user in all_users}
    
    for counter, future in enumerate(as_completed(futures), 1):
        try:
            df = future.result()
            df_list.append(df)
            print(counter, "/", len(all_users))
        except Exception as e:
            print(f"Error processing user {futures[future]}: {e}")

# Concatenate all DataFrames
combined_df = pd.concat(df_list, ignore_index=True)
combined_df.to_pickle(output_dir + "/all_user_active_content_df")

In [ ]:
# content df
content = combined_df[combined_df.action.isin([1, 2])]
content.to_pickle(output_dir +"/content_text_df.pkl")
# submission df
submissions = combined_df[combined_df.action == 1]
submissions['submission_text'] = np.where(
    submissions['selftext'].isna() | (submissions['selftext'] == "[removed]"),
    submissions['title'],
    submissions['title'] + " " + submissions['selftext']
)
submissions.to_pickle(output_dir +"/submission_text_df.pkl")


In [ ]:
# get passive actions (replies and parents)

# rus active timeframe
start_date = "2015-01-02"
end_date = "2018-04-11"

# Helper function to run in parallel
def get_user_df(user):
    df_slice = reddit_traj_construction.get_user_activity_in_timeframe(
        user,
        load_dir,
        start_date=start_date,
        end_date=end_date
    )
    # Only keep relevant rows and columns
    filtered = df_slice[df_slice.action.isin([4,5])].copy()

    # Ensure column exists
    if 'agree_prediction' not in filtered.columns:
        filtered['agree_prediction'] = None
    if 'url' not in filtered.columns:
        filtered['url'] = None
    if 'title' not in filtered.columns:
        filtered['title'] = None
    if 'selftext' not in filtered.columns:
        filtered['selftext'] = None
    if 'subreddit' not in filtered.columns:
        filtered['subreddit'] = None

    return filtered[["author", "id", "created_utc", "subreddit", "body", "selftext", "action", "url", "title", 'agree_prediction']].copy()

# Use all available cores
num_workers = multiprocessing.cpu_count()

df_list = []
with ProcessPoolExecutor(max_workers=num_workers) as executor:
    futures = {executor.submit(get_user_df, user): user for user in all_users}
    
    for counter, future in enumerate(as_completed(futures), 1):
        try:
            df = future.result()
            df_list.append(df)
            print(counter, "/", len(all_users))
        except Exception as e:
            print(f"Error processing user {futures[future]}: {e}")

# Concatenate all DataFrames
combined_df = pd.concat(df_list, ignore_index=True)
combined_df.to_pickle(output_dir + "/all_user_passive_content_df")